### Carregamento dos dados

Nesta etapa, a biblioteca `pandas` é importada e o arquivo CSV contendo os resultados dos experimentos é carregado em um `DataFrame`. Em seguida, são exibidas as primeiras linhas do conjunto de dados para verificar sua estrutura e os valores registrados.

In [2]:
import pandas as pd

df = pd.read_csv("../../results/init_duration_results.csv")

df.head()

,function_name,architecture,memory_size,init_duration_ms
0,lambda-01-arm-512mb,arm64,512,68.45
1,lambda-02-arm-512mb,arm64,512,87.61
2,lambda-03-arm-512mb,arm64,512,85.55
3,lambda-04-arm-512mb,arm64,512,83.72
4,lambda-05-arm-512mb,arm64,512,70.22


### Tratamento dos dados

Nesta etapa, a coluna `init_duration_ms`, que contém os valores de duração do cold start, é convertida para o formato numérico. Valores que não puderem ser convertidos são tratados como ausentes e posteriormente removidos do conjunto de dados, garantindo que apenas observações válidas sejam utilizadas na análise estatística.

In [13]:
df["init_duration_ms"] = pd.to_numeric(
    df["init_duration_ms"],
    errors="coerce"
)

df = df.dropna(subset=["init_duration_ms"])

### Estatísticas descritivas básicas

Nesta etapa, são calculadas as principais estatísticas descritivas da variável `init_duration_ms`. São obtidos a média aritmética, a mediana, o desvio padrão, o valor mínimo e o valor máximo das amostras coletadas, permitindo uma primeira caracterização dos tempos de inicialização observados.

In [15]:
media = df["init_duration_ms"].mean()
mediana = df["init_duration_ms"].median()
desvio_padrao = df["init_duration_ms"].std()
minimo = df["init_duration_ms"].min()
maximo = df["init_duration_ms"].max()

print(f"Média: {media:.2f} ms")
print(f"Mediana: {mediana:.2f} ms")
print(f"Desvio padrão: {desvio_padrao:.2f} ms")
print(f"Mínimo: {minimo:.2f} ms")
print(f"Máximo: {maximo:.2f} ms")

Média: 82.71 ms
Mediana: 83.69 ms
Desvio padrão: 11.58 ms
Mínimo: 46.16 ms
Máximo: 193.35 ms


### Resumo estatístico

A função `describe()` é utilizada para obter um resumo estatístico da variável `init_duration_ms`. Além das medidas calculadas anteriormente, são apresentados a quantidade de observações e os quartis de 25\%, 50\% e 75\%, permitindo observar a distribuição dos valores de duração do cold start.

In [12]:
df["init_duration_ms"].describe()

count    1000.000000
mean       82.714320
std        11.575002
min        46.160000
25%        76.862500
50%        83.695000
75%        87.617500
max       193.350000
Name: init_duration_ms, dtype: float64

### Análise por arquitetura

Nesta etapa, as amostras são agrupadas de acordo com a arquitetura da função Lambda (`arm64` e `x86_64`). Para cada arquitetura, são calculadas a quantidade de observações, a média, a mediana, o desvio padrão, o valor mínimo e o valor máximo do `Init Duration`, permitindo comparar estatisticamente os resultados obtidos entre as arquiteturas.

In [11]:
def p95(x):
    return x.quantile(0.95)

def p99(x):
    return x.quantile(0.99)

def cv(x):
    return x.std() / x.mean() * 100

def amp(x):
    return x.max() - x.min()

df.groupby("architecture")["init_duration_ms"].agg(
    ["count", "mean", "median", "std", "var", cv, "min", "max", amp, p95, p99]
)

,count,mean,median,std,var,cv,min,max,amp,p95,p99
architecture,,,,,,,,,,,
arm64,500,81.02362,83.415,12.512679,156.567127,15.443248,46.16,193.35,147.19,94.1095,122.5649
x86_64,500,84.40502,84.120,10.292440,105.934312,12.194108,60.63,155.09,94.46,100.9255,112.0636


### Análise por arquitetura e configuração de memória

Nesta etapa, as amostras são agrupadas pela combinação entre arquitetura e tamanho de memória, permitindo analisar cada cenário experimental individualmente. Para cada configuração, são calculadas medidas de tendência central e dispersão, incluindo média, mediana, desvio padrão e variância, além dos valores mínimo e máximo. Também são calculados o coeficiente de variação (CV) e os percentis P95 e P99, utilizados para caracterizar a variabilidade e o comportamento das maiores durações de cold start.

In [3]:
def p95(x):
    return x.quantile(0.95)

def p99(x):
    return x.quantile(0.99)

def cv(x):
    return x.std() / x.mean() * 100

def amp(x):
    return x.max() - x.min()

df.groupby(
    ["architecture", "memory_size"]
)["init_duration_ms"].agg(
    ["count", "mean", "median", "std", "var", cv, "min", "max", amp, p95, p99]
)


count     mean  median        std         var  \
architecture memory_size                                                  
arm64        128            100  79.7077  82.790  11.355452  128.946291   
             512            100  80.7603  83.010  11.049496  122.091365   
             1024           100  81.5906  83.405  15.725935  247.305030   
             2048           100  82.3873  83.760  11.918989  142.062295   
             3008           100  80.6722  83.690  12.025372  144.609563   
x86_64       128            100  87.0381  86.470  10.571016  111.746369   
             512            100  85.2949  84.210  11.625643  135.155577   
             1024           100  80.8548  81.630   8.837988   78.110027   
             2048           100  84.2441  84.100   8.925111   79.657598   
             3008           100  84.5932  84.895  10.425250  108.685832   

                                 cv    min     max     amp       p95       p99  
architecture memory_size                                                        
arm64        128          14.246368  49.43  112.65   63.22   91.9575  110.3532  
             512          13.681841  46.16  128.65   82.49   90.6680  109.8598  
             1024         19.274199  47.28  193.35  146.07   94.1455  123.7530  
             2048         14.467022  64.84  138.00   73.16   95.5565  122.7144  
             3008         14.906463  49.04  145.92   96.88   92.6035   97.6080  
x86_64       128          12.145274  60.63  117.14   56.51  105.5935  112.1108  
             512          13.629939  65.11  155.09   89.98  100.3810  112.8467  
             1024         10.930690  61.66  111.87   50.21   95.8610  101.1384  
             2048         10.594345  69.43  123.29   53.86   99.7125  104.6087  
             3008         12.323981  67.47  118.91   51.44  103.0345  107.8220